# 🔎 VeriFile AI — Deteksi Otomatis Bukti Transfer & Struk Pembayaran

**Sistem Forensik Single-Upload Bukti Transfer**
Cukup unggah gambar bukti transfer (JPG, PNG, WebP) — sistem akan langsung mendeteksi apakah bukti transfer asli, terindikasi rekayasa **Generative AI (C2PA / SynthID)**, **software editor grafis (Photoshop/Canva)**, **foto ulang layar HP (recapture)**, serta mengekstraksi data struk dan memeriksa **konsistensi nominal** secara otomatis.

**Cara menggunakan di Google Colab:**
1. Jalankan cell secara berurutan.
2. Mode forensik inti berjalan 100% lokal tanpa memerlukan GPU.
3. (Opsional GPU) Pilih *Runtime > Change runtime type > T4 GPU* jika ingin mengaktifkan Gemma 4 Multimodal.

In [ ]:
import os, sys, shutil, subprocess
print("Python:", sys.version.split()[0])
try:
    import torch
    print("GPU aktif:", torch.cuda.is_available())
    if torch.cuda.is_available():
        print("Device:", torch.cuda.get_device_name(0))
except ImportError:
    print("Torch belum diinstal; mode forensik inti tetap berjalan penuh.")

## 1. Clone Repositori VeriFile-AI
Mengambil kode sumber terbaru langsung dari repositori GitHub.

In [ ]:
!git clone https://github.com/alifakbxr/VeriFile-AI.git /content/VeriFile-AI 2>/dev/null || (cd /content/VeriFile-AI && git pull)
import os, sys
if os.path.exists("/content/VeriFile-AI"):
    os.chdir("/content/VeriFile-AI")
    if "/content/VeriFile-AI" not in sys.path:
        sys.path.insert(0, "/content/VeriFile-AI")
print("Direktori kerja:", os.getcwd())

## 2. Instal Dependensi, RapidOCR, dan Jalankan Pengujian
Menginstal `requirements-ocr.txt` (Pillow, OpenCV, PyMuPDF, defusedxml, RapidOCR, ONNX Runtime) dan mengeksekusi 34 automated unit tests.

In [ ]:
%pip -q install -r requirements-ocr.txt
!python demo/create_samples.py
!PYTHONPATH=. pytest -v

## 3. Demo Uji Single-Upload Bukti Transfer Asli vs Rekayasa AI
Menguji deteksi langsung pada sampel data bukti transfer (`demo/TEST DATA/`):

In [ ]:
from verifile.forensics import analyze_file
from pathlib import Path

orig_path = Path("demo/TEST DATA/202610300859170551-1793419781221.jpg")
fake_path = Path("demo/TEST DATA/202610300859170551-179341978122122.jpg")

print("=== 1. UJI SINGLE-UPLOAD BUKTI ASLI ===")
orig_rep, _, _ = analyze_file(orig_path)
print(f"Verdict: {orig_rep.get('verdict_title')}")
print(f"Kategori: {orig_rep.get('verdict_category')} (Risiko: {orig_rep.get('risk_level')})")
print(f"Bank: {orig_rep.get('receipt_data', {}).get('bank')}")
print(f"Nominal: {orig_rep.get('receipt_data', {}).get('amounts')}")

print("\n=== 2. UJI SINGLE-UPLOAD BUKTI REKAYASA GENERATIVE AI (FAKE) ===")
fake_rep, _, _ = analyze_file(fake_path)
print(f"Verdict: {fake_rep.get('verdict_title')}")
print(f"Kategori: {fake_rep.get('verdict_category')} (Risiko: {fake_rep.get('risk_level')})")
print(f"Bank: {fake_rep.get('receipt_data', {}).get('bank')}")
print(f"Nominal: {fake_rep.get('receipt_data', {}).get('amounts')}")
print("Temuan Forensik:")
for f in fake_rep.get('forensics', {}).get('findings', []):
    if f.get('severity') in ['alert', 'review']:
        print(f" - [{f['severity'].upper()}] {f['title']}: {f['detail']}")

## 4. (Opsional) Aktifkan Gemma 4 Multimodal di Colab GPU
Hanya jika runtime GPU (T4) aktif dan HF_TOKEN tersedia di Colab Secrets.

In [ ]:
ENABLE_GEMMA = False
if ENABLE_GEMMA:
    import torch
    if not torch.cuda.is_available():
        raise RuntimeError("Pilih T4 GPU di Runtime > Change runtime type terlebih dahulu.")
    # Pastikan bitsandbytes >= 0.46.1 dan accelerate terinstal untuk kuantisasi 4-bit
    get_ipython().system("pip install -q -U 'bitsandbytes>=0.46.1' 'accelerate>=1.1.0' transformers")
    subprocess.run([sys.executable, "-m", "pip", "install", "-U", "-r", "requirements-gemma.txt"], check=True)
    os.environ["GEMMA_4BIT"] = "1"
    os.environ["GEMMA_MODEL_ID"] = "google/gemma-4-E2B-it"
    try:
        from google.colab import userdata
        hf_token = userdata.get("HF_TOKEN")
        if hf_token:
            from huggingface_hub import login
            login(token=hf_token, add_to_git_credential=False)
            print("Hugging Face authentication configured.")
    except Exception as e:
        print("Colab secret HF_TOKEN belum diset:", type(e).__name__)
    print("Gemma siap digunakan ketika opsi checkbox diaktifkan di UI.")
else:
    print("Skip Gemma. Forensic engine inti berjalan penuh tanpa GPU.")

## 5. Jalankan Demo Gradio (Mendukung Multi-Upload Bukti Transfer)

Meluncurkan antarmuka web interaktif yang mendukung **multi-upload gambar bukti transfer sekaligus**. Pengguna cukup mengunggah satu atau beberapa gambar bukti transfer — sistem akan langsung mendeteksi rekayasa Generative AI (C2PA/SynthID), software editor, foto layar HP, dan mengekstrak rincian struk untuk setiap berkas secara otomatis tanpa perlu memilih file pembanding asli atau palsu.


In [ ]:
SHARE_PUBLIC_DEMO = False

import os
auth = None
if os.getenv("VERIFILE_AUTH_USER") and os.getenv("VERIFILE_AUTH_PASS"):
    auth = (os.environ["VERIFILE_AUTH_USER"], os.environ["VERIFILE_AUTH_PASS"])

import app as verifile_app
verifile_app.build_app().launch(
    server_name="0.0.0.0",
    share=SHARE_PUBLIC_DEMO,
    auth=auth,
    debug=True,
    max_file_size="20mb"
)